In [8]:
import pandas as pd
from pathlib import Path

# =========================
# PATHS
# =========================
HOMES_DIR = Path("E:/IDEAL/processed/hourly_per_home_filled")
WEATHER_PATH = Path("E:/IDEAL/auxiliarydata/weather/edinburgh_hourly_temperature.csv")

OUTPUT_DIR = Path("E:/IDEAL/processed/hourly_per_home_final")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# LOAD WEATHER
# =========================
weather = pd.read_csv(WEATHER_PATH, parse_dates=["timestamp"])

# Κρατάμε μόνο τις στήλες που χρειαζόμαστε + σιγουρευόμαστε για uniqueness
weather = weather[["timestamp", "external_temperature"]].drop_duplicates(subset=["timestamp"])

# Προαιρετικό: έλεγχος ότι δεν έχουμε NaN στο weather
print("Weather rows:", len(weather))
print("Weather NaN:", weather["external_temperature"].isna().sum())
print("Weather range:", weather["timestamp"].min(), "->", weather["timestamp"].max())

# =========================
# MERGE PER HOME
# =========================
summary = []

for csv_file in sorted(HOMES_DIR.glob("home*_hourly.csv")):
    home_id = csv_file.stem.replace("_hourly", "")
    df = pd.read_csv(csv_file, parse_dates=["timestamp"])

    # Merge με βάση το timestamp (left join: κρατάμε όλα τα timestamps κατανάλωσης)
    df = df.merge(weather, on="timestamp", how="left", suffixes=("", "_wx"))

    # Αν υπήρχε παλιά external_temperature στήλη, την αντικαθιστούμε με τη νέα
    # (στο δικό σου pipeline ήταν NaN, αλλά το κάνουμε σωστά/ασφαλώς)
    if "external_temperature_wx" in df.columns:
        # Σε περίπτωση που είχε ήδη external_temperature από πριν
        if "external_temperature" in df.columns:
            df.drop(columns=["external_temperature"], inplace=True)
        df.rename(columns={"external_temperature_wx": "external_temperature"}, inplace=True)

    # Στρογγυλοποίηση external temperature σε 1 δεκαδικό (προαιρετικό αλλά ωραίο)
    df["external_temperature"] = df["external_temperature"].round(1)

    # Μετράμε NaN που έμειναν (αν κάποια timestamps είναι εκτός weather range)
    nan_ext = df["external_temperature"].isna().sum()

    out_path = OUTPUT_DIR / csv_file.name
    df.to_csv(out_path, index=False)

    summary.append({
        "home_id": home_id,
        "rows": len(df),
        "nan_external_temperature": nan_ext
    })

summary_df = pd.DataFrame(summary).sort_values("nan_external_temperature", ascending=False)
summary_df

Weather rows: 19008
Weather NaN: 0
Weather range: 2016-07-01 00:00:00 -> 2018-08-31 23:00:00


,home_id,rows,nan_external_temperature
0,home100,11083,0
1,home101,10917,0
2,home102,10198,0
3,home105,8709,0
4,home106,9843,0
...,...,...,...
249,home94,11501,0
250,home96,8468,0
251,home97,9575,0
252,home98,11290,0
